# Apprentissage non-supervisé : partitionnement hiérarchique et sélection de modèle

# Table of contents
1. [Partitionnement agglomératif](#part1)
1. [Sélection de modèle](#part2)


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# %matplotlib inline
sns.set()

from matplotlib import cm
from scipy.stats import multivariate_normal

In [ ]:
def plotXY(X, Y, legend=True):
    """
        Scatter points with a color for each class.
        Input:
            X and Y may be:
            - two numpy arrays with two columns; each array is the data matrix for a class (works only for
            two classes).
            - a numpy array with two columns (the data matrix) and the vector of labels (works for many classes).
    """    
    if Y.ndim > 1:
        X1 = X
        X2 = Y
        XX = np.concatenate((X, Y), axis=0)
        YY = np.concatenate((np.ones(X.shape[0]), -np.ones(Y.shape[0])))
    else:
        XX = X
        YY = Y
    for icl, cl in enumerate(np.unique(YY)):
        plt.scatter(XX[YY==cl, 0], XX[YY==cl, 1], label='Class {0:d}'.format(icl+1))
    plt.axis('equal')
    if legend:
        plt.legend()
        
def covariance(sigma1=1., sigma2=1., theta=0.):
    """
        Covariance matrix with eigenvalues sigma1 and sigma2, rotated by the angle theta.
    """
    rotation = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta), np.cos(theta)]])
    cov = np.array([[sigma1, 0.],
                   [0, sigma2]])
    return rotation.dot(cov.dot(rotation.T))    

def sample_gm(weights, means, covariances, size=1):
    """Sample points from a Gaussian mixture model specified by the weights, the means
    and the covariances. These three parameters are lists."""
    X = None
    p = np.random.multinomial(1, weights, size=size)
    for (m, c, i) in zip(means, covariances, p.T):
        Y = np.random.multivariate_normal(m, c, size=size)
        if X is None:
            X = Y.copy()
        else:
            X[i==1] = Y[i==1]
    return X

# Partitionnement agglomératif <a id="part1"></a>


<div class="alert alert-block alert-info">

Nous cherchons à comparer les quatre variantes du partitionnement agglomératif : <br>    
1. single linkage ; <br>
2. complete linkage ; <br>
3. average linkage ; <br>
4. Ward's method. <br>
    
À l'aide de la fonction `make_clusters`, générer un échantillon de taille 200 et l'afficher.

<!-- <br> -->
</div>

In [ ]:
def make_rect(center=[0, 0], width=2, size=1):
    """Random points in a square.
    """
    X = (np.random.rand(size, 2) + np.asarray(center) - np.r_[0.5, 0.5])
    X[:, 0] *= width
    return X

def make_clusters(size=1, width=4):
    """Random points in 2 close rectangles.
    """
    ind = np.sort(np.random.randint(size//5, high=4*size//5, size=1))[0]
    sizes = [ind, size-1-ind]
    centers = [[0, 1], [0, -1]]
    return np.concatenate([make_rect(c, size=s, width=width) for c, s in zip(centers, sizes)])

In [ ]:
# Answer
X = make_clusters(size=200)

<div class="alert alert-block alert-info">

À l'aide de la classe <a href="https://scikit-learn.org/stable/modules/generated/sklearn.cluster.AgglomerativeClustering.html">AgglomerativeClustering</a>, afficher le résultat obtenu pour *single linkage*.

<!-- <br> -->
</div>

In [ ]:
# Answer
from sklearn.cluster import AgglomerativeClustering

single_linkage_clt = AgglomerativeClustering(n_clusters=2, linkage="single")
y = single_linkage_clt.fit_predict(X)

plotXY(X, y)

<div class="alert alert-block alert-info">

Nous souhaitons comparer le partitionnement résultant pour : <br>    
- 4 ou 2 groupes ; <br>
- les quatre variantes du partitionnement agglomératif.<br>
    
Pour ce faire, compéter la fonction suivante et l'appliquer au jeu de données.
    Analyser les résultats.

<!-- <br> -->
</div>

In [ ]:
# Answer
def compare_agg(X, Clt=AgglomerativeClustering, n_clusters=[2], methods=["single"], figsize=(15, 2)):
    for k in n_clusters:
        plt.figure(figsize=figsize)  # Create figure with appropriate size
        for it, method in enumerate(methods):
            # Fit and predict
            # To do
            clt = Clt(n_clusters=k, linkage=method)
            y = clt.fit_predict(X)

            # End to do

            plt.subplot(1, len(methods), it+1)
            # Scatter points with one color for each cluster
            # To do

            plotXY(X, y)

            # End to do
            plt.title(method)

In [ ]:
# Answer
compare_agg(X, n_clusters=[2, 4], methods=["single", "complete", "average", "ward"])

**Answer:**
…

<div class="alert alert-block alert-info">

Répéter l'expérience avec un jeu de données rectangulaire.

<!-- <br> -->
</div>

In [ ]:
# Answer
X_rect = make_rect(center=[0, 0], width=2, size=200)

compare_agg(X_rect, n_clusters=[2, 4], methods=["single", "complete", "average", "ward"])

**Answer:**
…

<div class="alert alert-block alert-info">

Répéter l'expérience avec ce jeu de données, issu d'un GMM.

<!-- <br> -->
</div>

In [ ]:
# Dataset
(weights, means, covariances) = ([0.33, 0.33, 0.34],
                                 [[0, 0], [5, 0], [2, -5]],
                                 [(1, 1, 0), (1, 1, 0), (1, 1, 0)])

X_gmm = sample_gm(weights, means, [covariance(*c) for c in covariances], size=200)

In [ ]:
# Answer
compare_agg(X_gmm, n_clusters=[2, 4], methods=["single", "complete", "average", "ward"])

**Answer:**
…

# Sélection de modèle <a id="part2"></a>


<div class="alert alert-block alert-info">

Voici un jeu de données et une fonction permettant de calculer l'inertie intraclasse (pour une matrice de données `X` et des classes prédites `y`).

<!-- <br> -->
</div>

In [ ]:
from sklearn.datasets import make_blobs, make_moons

blobs = make_blobs(n_samples=300, centers=10, cluster_std=0.7)[0]
moons1 = make_moons(n_samples=100, noise=0.05)[0]*3
moons1[:, 0] += 15
moons2 = make_moons(n_samples=100, noise=0.05)[0]*3
moons2[:, 1] -= 15
noise = np.random.rand(20, 2) * 30 - 15

X = np.concatenate((blobs,moons1, moons2, noise))

In [ ]:
def intraclass_inertia(X, y):
    Z = X.copy()
    for id_class in range(y.min(), y.max()+1):
        Z[y==id_class] -= Z[y==id_class].mean()
    return np.linalg.norm(Z, 'fro')**2 / Z.shape[0]

<div class="alert alert-block alert-info">

Afficher le résultat d'un partitionnement agglomératif des données avec 14 groupes.

<!-- <br> -->
</div>

In [ ]:
# Answer

compare_agg(X, n_clusters=[7], methods=["single", "complete", "average", "ward"])

<div class="alert alert-block alert-info">

Tracer les courbes de l'inertie et du <a href="http://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score">coefficient silhouette</a> en fonction du nombre de groupes.

<!-- <br> -->
</div>

In [ ]:
# Answer

<div class="alert alert-block alert-info">

Afficher le résultat du partitionnement obtenu avec le nombre de groupes condisant au plus haut <a href="http://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html#sklearn.metrics.silhouette_score">coefficient silhouette</a>.

<!-- <br> -->
</div>

In [ ]:
# Answer